
# Titanic Survival Prediction – Full Data Science Project

Workflow
1. Data loading
2. Data inspection
3. Exploratory Data Analysis
4. Feature engineering
5. Preprocessing
6. Model training
7. Cross validation
8. Feature importance
9. Model comparison
10. Ensemble prediction
11. Kaggle submission


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb


## Load Data

In [ ]:

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

train_df.head()


## Data Inspection

In [ ]:

train_df.info()


In [ ]:

train_df.describe()


In [ ]:

train_df.isnull().sum()


## Exploratory Data Analysis

In [ ]:

sns.countplot(data=train_df, x="Sex", hue="Survived")
plt.show()


In [ ]:

sns.countplot(data=train_df, x="Pclass", hue="Survived")
plt.show()


In [ ]:

sns.histplot(data=train_df, x="Age", hue="Survived", bins=30)
plt.show()


## Feature Engineering

In [ ]:

train_df["FamilySize"] = train_df["SibSp"] + train_df["Parch"] + 1
test_df["FamilySize"] = test_df["SibSp"] + test_df["Parch"] + 1

train_df["IsAlone"] = (train_df["FamilySize"] == 1).astype(int)
test_df["IsAlone"] = (test_df["FamilySize"] == 1).astype(int)


In [ ]:

train_df["Title"] = train_df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)
test_df["Title"] = test_df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)


In [ ]:

train_df["Age"] = train_df["Age"].fillna(train_df["Age"].median())
test_df["Age"] = test_df["Age"].fillna(test_df["Age"].median())

train_df["Embarked"] = train_df["Embarked"].fillna(train_df["Embarked"].mode()[0])
test_df["Embarked"] = test_df["Embarked"].fillna(test_df["Embarked"].mode()[0])

test_df["Fare"] = test_df["Fare"].fillna(test_df["Fare"].median())


## Encode Categorical Variables

In [ ]:

categorical_cols = ["Sex", "Embarked", "Title"]

encoder = LabelEncoder()

for col in categorical_cols:
    train_df[col] = encoder.fit_transform(train_df[col])
    test_df[col] = encoder.fit_transform(test_df[col])


## Feature Selection

In [ ]:

features = [
    "Pclass","Sex","Age","SibSp","Parch","Fare",
    "Embarked","FamilySize","IsAlone","Title"
]

X = train_df[features]
y = train_df["Survived"]
X_test = test_df[features]


## RandomForest Model

In [ ]:

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

rf.fit(X,y)


## Cross Validation

In [ ]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(rf, X, y, cv=cv)

print("CV Accuracy:", scores.mean())


## Feature Importance

In [ ]:

importances = rf.feature_importances_

feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)

feat_imp.plot(kind="bar")
plt.show()


## LightGBM Model

In [ ]:

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05
)

lgb_model.fit(X,y)


## Model Comparison

In [ ]:

rf_score = cross_val_score(rf, X, y, cv=cv).mean()
lgb_score = cross_val_score(lgb_model, X, y, cv=cv).mean()

print("RandomForest:", rf_score)
print("LightGBM:", lgb_score)


## Ensemble Prediction

In [ ]:

rf_pred = rf.predict_proba(X_test)[:,1]
lgb_pred = lgb_model.predict_proba(X_test)[:,1]

ensemble_pred = (rf_pred + lgb_pred) / 2

final_pred = (ensemble_pred > 0.5).astype(int)


## Submission File

In [ ]:

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": final_pred
})

submission.to_csv("submission.csv", index=False)

submission.head()
